# 🧪 Test Notebook: Schema Mapper Node
Kiểm thử các chức năng của Schema Mapper:
1. Trích xuất cột giá trị hữu dụng (`useful_columns`)
2. Nhận diện danh mục con (`sub_sections`), tính `range` và `total_value`
3. Làm giàu mô tả cột bằng LLM (`_enrich_column_descriptions`)
4. Chạy toàn diện `schema_mapper_node`

In [ ]:
import os
import sys
import pandas as pd
from pathlib import Path

# Thêm root dự án vào sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pipeline.src.config import config
from pipeline.src.nodes.schema_mapper import (
    _get_columns_from_table,
    _find_label_column,
    _find_value_column,
    _extract_useful_columns,
    _extract_sub_sections,
    _enrich_column_descriptions,
    schema_mapper_node,
    METADATA_HEADER_COLUMNS,
)
print("✅ Đã nạp thành công các hàm từ Schema Mapper!")

### 1. Tạo dữ liệu tài chính giả lập có cấu trúc Section & Cột số để kiểm thử

In [ ]:
# Tạo DataFrame mẫu mô phỏng Báo cáo tài chính với Section và Header số
sample_financial_data = {
    "Ma_Doanh_Nghiep": ["VNM"] * 10,
    "Nam_Tai_Chinh": ["2023"] * 10,
    "0": ["TÀI SẢN", "", "I. TÀI SẢN NGẮN HẠN", "Tiền và tương đương tiền", "Đầu tư tài chính ngắn hạn", "Các khoản phải thu ngắn hạn", "", "II. TÀI SẢN DÀI HẠN", "Tài sản cố định", "Bất động sản đầu tư"],
    "1": ["31/12/2023", "", "35000000", "5000000", "15000000", "15000000", "", "25000000", "20000000", "5000000"],
    "2": ["01/01/2023", "", "30000000", "4000000", "14000000", "12000000", "", "22000000", "18000000", "4000000"]
}
df_test = pd.DataFrame(sample_financial_data)
test_csv_path = PROJECT_ROOT / "pipeline" / "data" / "test_financial_statement.csv"
test_csv_path.parent.mkdir(parents=True, exist_ok=True)
df_test.to_csv(test_csv_path, index=False)
print(f"✅ Đã tạo file CSV mẫu tại: {test_csv_path}")
df_test

### 2. Kiểm thử `_extract_useful_columns()`
- Tự động nhận diện cột chứa số
- Duyệt hàng header để đổi tên các cột có tên là số ('1' -> '31/12/2023', '2' -> '01/01/2023')

In [ ]:
label_col = _find_label_column(list(df_test.columns))
metadata_cols = set(METADATA_HEADER_COLUMNS)
useful_cols = _extract_useful_columns(df_test, label_col, metadata_cols)
print(f"Cột nhãn phát hiện: '{label_col}'")
print(f"Danh sách useful_columns trích xuất được:")
for uc in useful_cols:
    print(" -", uc)

assert len(useful_cols) == 2, f"Kỳ vọng 2 cột giá trị, nhận được {len(useful_cols)}"
assert useful_cols[0]["column_name"] == "31/12/2023", f"Cột 1 phải được map thành '31/12/2023', thực tế: {useful_cols[0]['column_name']}"
assert useful_cols[1]["column_name"] == "01/01/2023", f"Cột 2 phải được map thành '01/01/2023', thực tế: {useful_cols[1]['column_name']}"
print("✅ Test 1: _extract_useful_columns PASS!")

### 3. Kiểm thử `_extract_sub_sections()`
- Nhận diện các section ngăn cách bởi hàng trống
- Trích xuất `total_value` trên hàng section
- Xác định chính xác `range` [start, end] các hàng con

In [ ]:
sub_sections = _extract_sub_sections(df_test, label_col, metadata_cols)
print("Danh sách sub_sections trích xuất được:")
for sec in sub_sections:
    print(" -", sec)

assert len(sub_sections) == 2, f"Kỳ vọng 2 sections, nhận được {len(sub_sections)}"
sec1 = sub_sections[0]
assert sec1["section_name"] == "I. TÀI SẢN NGẮN HẠN"
assert sec1["total_value"] == 35000000.0
assert sec1["range"] == [3, 5], f"Range của Section 1 phải là [3, 5], thực tế: {sec1['range']}"

sec2 = sub_sections[1]
assert sec2["section_name"] == "II. TÀI SẢN DÀI HẠN"
assert sec2["total_value"] == 25000000.0
assert sec2["range"] == [8, 9], f"Range của Section 2 phải là [8, 9], thực tế: {sec2['range']}"

print("✅ Test 2: _extract_sub_sections PASS!")

### 4. Kiểm thử toàn diện `schema_mapper_node()`

In [ ]:
mock_state = {
    "user_query": "Tài sản ngắn hạn của VNM năm 2023",
    "parsed_query": {
        "ten_cong_ty": "VNM",
        "so_nam": ["2023"],
        "noi_dung": "Tài sản ngắn hạn",
        "tieu_chi_phu": "31/12/2023"
    },
    "discovered_tables": [
        {
            "csv_path": str(test_csv_path),
            "Ten_Bang": "Bảng Cân đối kế toán VNM 2023",
            "Nam_Tai_Chinh": "2023"
        }
    ]
}

res_state = schema_mapper_node(mock_state, config)
print("\nKết quả schema_mapper_node output:")
import json
print(json.dumps(res_state.get("schema"), indent=2, ensure_ascii=False))
print(f"Column Mapping: {res_state.get('column_mapping')}")

assert "schema" in res_state
assert len(res_state["schema"]["useful_columns"]) == 2
assert len(res_state["schema"]["sub_sections"]) == 2
print("✅ Test 3: schema_mapper_node integration PASS!")